# v3.5 link C — VEi + head crop against re-pose + head crop

Four new arms over the **51 pairs either failure record marks hard** (`v35_linkC.csv`), seeds **46/47/48**, self-hosted klein on this A100.

| arm | call 1 | head off | ankle cut | computed here |
|---|---|---|---|---|
| `VEi` | the lock's `Q3` — mannequin head **and** re-pose | no | no | **no** — reused from iron man 2 |
| `BC` | v3.1's incumbent: bald pass → V2 crop | the V2 cropper | no | **no** — reused from iron man 2 |
| `VEic` | the lock's `Q3` | a crop | no | yes |
| `M1qc` | `Q3` **with the mannequin sentence deleted** | a crop | no | yes |
| `VEica` | as `VEic` | a crop | **yes** | yes |
| `M1qca` | as `M1qc` | a crop | **yes** | yes |

An arm name is read, not looked up: base (`VEi` | `M1q`) + `c` head crop + `a` ankle cut. `VEic` and `M1qc` differ by exactly one deleted sentence in call 1; each `a` arm is paired with its cut-less twin at the same seed, so the ankle cut is the only difference between them.

**The set.** Two failure records exist for this matrix and they overlap on only 11 pairs, so link C runs the union of 51:

- **20 pairs** the **v3.4 lock** failed and v3.3 did not — the reviewer's per-cell verdict on `VEi` (`v34_im2_truth.json`, 600 cells judged).
- **20 pairs** **v3.3** failed and the lock does not — carrying the `F1`–`F4` class.
- **11 pairs** both records mark.

There is **no per-cell record of a correctly built `BC` failing** on this matrix: the reviewer judged the `VEi` arm only. The nearest is v3.3's *both arms failed* verdict, and that BC was BCA4-class — already inside the v3.3 half of this union by construction. (The four named `BC_klein` failures of v3.0 are on a different fold and are not in this matrix.)

**Order of operations, which is not the obvious one:**

    call 1 ──▶ white-margin re-crop ──▶ HEAD CROP ──▶ [ankle cut] ──▶ SR to ~1 MP ──▶ call 2

The crops go **before** the SR pass. Cropping afterwards would take the reference back below 1 MP and break the rule link H bought — what conditioning contributes is bounded by its **token footprint** in call 2 (v3.4 SOLUTION §5, rule 3). A cropped reference is a smaller image and has to be re-floated, or the arm tests two changes at once.

Neither crop is new code. The head crop is `ironman_bc_crop.crop_bc`, the call that makes the `BC` references — it fires on a **mannequin** head as well as a real one, because the cranium path was built for BC's *bald* frames and takes head shape from the human parser, head extent from pose landmarks. The ankle cut is `run_ironman.ankle_cut`, v3.3's, verbatim; **v3.4 removed it at the lock**, so it is reopened here as its own variable, and it is a no-op where no ankles are in frame.

**New klein calls: 33 references + 612 edits ≈ 645** (~50 min on an A100), because the lock's pre-SR references and both baselines' cells come off Drive rather than being redrawn. Set `ARMS = ("VEic", "M1qc")` in cell 1 to drop the ankle-cut arms and halve it.

**One session.** Needs the two iron-man 2 zips on Drive: `v3_runs/v34_ironman2_*.zip` and `v3_runs/v34_ironman2_bc_*.zip`.

In [ ]:
# 1 · settings
A100_USD_PER_HOUR = 0.689     # CAD/h at 5.3 CU/h x CAD 0.13/CU; edit if your rate differs
SEEDS = [46, 47, 48]          # the iron-man 2 seeds: every lock cell pairs with a verdict
ARMS = ("VEic", "M1qc", "VEica", "M1qca")   # drop the two 'a' arms to halve the run
MATRIX = "v35_linkC.csv"      # 51 pairs: the union of the v3.4 lock's and v3.3's failures
DRIVE_PROJECT_DIR = "Side projects and shi"

In [ ]:
# 2 · Drive: the HF cache that holds klein, and the model dir
import os
from google.colab import drive
drive.mount('/content/drive')
MYDRIVE = '/content/drive/MyDrive'; BASE = os.path.join(MYDRIVE, DRIVE_PROJECT_DIR)
KLEIN = 'models--black-forest-labs--FLUX.2-klein-4B'
candidates = [os.path.join(MYDRIVE, 'hf_cache'), os.path.join(BASE, 'tryon_models', 'hf_cache'), os.path.join(BASE, 'hf_cache')]
found = [c for c in candidates if os.path.isdir(os.path.join(c, 'hub', KLEIN))]
os.environ['HF_HOME'] = found[0] if found else candidates[0]
os.environ['V3_MODEL_DIR'] = os.path.join(BASE if os.path.isdir(BASE) else MYDRIVE, 'v3_models')
os.makedirs(os.environ['HF_HOME'], exist_ok=True); os.makedirs(os.environ['V3_MODEL_DIR'], exist_ok=True)
# the V2 cropper's BiRefNet and human parser default to CPU - a 4-core-laptop decision
# from V2, and every V2 number on record was measured that way. Opt them onto this GPU;
# cell 6 validates the result against the refs of record before the arm is trusted.
os.environ['V2_ORT_GPU'] = '1'
print('HF_HOME      ', os.environ['HF_HOME'], '(klein cached)' if found else '(klein will download, ~9 GB)')
print('V3_MODEL_DIR ', os.environ['V3_MODEL_DIR'])

In [ ]:
# 3 · install; pull the bundle from GitHub (public, branch v3.3-lock)
# onnxruntime and onnxruntime-gpu cannot coexist: leaving the CPU wheel installed makes
# CUDAExecutionProvider vanish from get_available_providers(), the cropper falls back to
# CPU exactly as designed, and the only symptom is that it is slow. Remove it first.
!pip -q uninstall -y onnxruntime >/dev/null 2>&1
!pip -q install -U diffusers transformers accelerate sentencepiece protobuf mediapipe onnxruntime-gpu opencv-contrib-python-headless
!cd /content && rm -rf v35 && wget -q -O v35_bundle.zip https://github.com/101011101/magichour_takehome/raw/v3.3-lock/v35_linkC_bundle.zip && unzip -qo v35_bundle.zip -d v35
%cd /content/v35
import os, torch, onnxruntime as ort
for f in ('realesr-general-x4v3.pth', 'lib/run_v35_linkC.py', 'lib/ironman_bc_crop.py',
          'lib/phase3_variants.py', 'v35_linkC.csv', 'v35_failures.csv'):
    assert os.path.exists(f), f'bundle incomplete: {f}'
os.makedirs('run/inputs', exist_ok=True)
print(torch.cuda.get_device_name(0), '|', ort.get_available_providers()[:2])
assert 'CUDAExecutionProvider' in ort.get_available_providers(), (
    'onnxruntime cannot see CUDA - the cropper would silently run on CPU (~80s an image). '
    'Restart the runtime after this cell and run it again.')

In [ ]:
# 4 · the iron-man 2 run off Drive: crops, the lock's pre-SR references, and both
#     baselines' cells. Nothing here is recomputed - it is the record this arm is
#     measured against, and redrawing it would break the pairing with the verdicts.
import glob, zipfile as zf, csv
rows = list(csv.DictReader(open(MATRIX)))
pairs = {r['set_id'] for r in rows}
stems = {r['person'] for r in rows} | {r['garment'] for r in rows}
garments = {r['garment'] for r in rows}

WANT = ({f'inputs/{s}.jpg' for s in stems}
        | {f'inputs/{g}__A4.jpg' for g in garments}
        | {f'refs/{g}__{t}.jpg' for g in garments for t in ('VEi_small', 'VEi', 'BC', 'bald')}
        | {f'gen/{sid}__{a}__s{s}.jpg' for sid in pairs for a in ('VEi', 'BC') for s in SEEDS})

def pick(pat, exclude=None):
    zs = [z for z in sorted(glob.glob(os.path.join(BASE, 'v3_runs', pat)))
          if not (exclude and exclude in os.path.basename(z))]
    assert zs, f'no {pat} on Drive under v3_runs/' + (f' (excluding {exclude})' if exclude else '')
    return zs[-1]

# '_bc_' sorts AFTER the digits, so a bare v34_ironman2_*.zip glob resolves to the BC zip
zips = [pick('v34_ironman2_*.zip', exclude='_bc_'), pick('v34_ironman2_bc_*.zip')]
assert len(set(zips)) == 2, f'both patterns resolved to the same file: {zips}'

got = 0
for zp in zips:
    with zf.ZipFile(zp) as z:
        members = [n for n in z.namelist() if n in WANT]
        z.extractall('run', members=members); got += len(members)
    print(f'  {os.path.basename(zp)}: {len(members)} files')

n_small = len(glob.glob('run/refs/*__VEi_small.jpg')); n_bc = len(glob.glob('run/refs/*__BC.jpg'))
n_cells = len(glob.glob('run/gen/*__VEi__*.jpg')) + len(glob.glob('run/gen/*__BC__*.jpg'))
print(f'{got} files | VEi_small {n_small}/{len(garments)} · BC refs {n_bc}/{len(garments)} · '
      f'baseline cells {n_cells}/{2 * len(rows) * len(SEEDS)}')
assert n_small == len(garments), "the lock's pre-SR references are incomplete - link C would redraw them"


In [ ]:
# 5 · load klein once, timed
import sys; sys.path.insert(0, 'lib')
import klein_local as K
K.load(); K.info()

In [ ]:
# 6 · one garment first: does the head come off a MANNEQUIN head?
#     The cropper is validated against the refs of record before it is trusted, then the
#     crop is run on a VEi reference - the case that has never been proved, because BC
#     only ever cropped a bald photograph, never a generated mannequin head.
import cv2, numpy as np, glob
import ironman_bc_crop as C
from IPython.display import display
from PIL import Image

def side_by_side(imgs, h=460):
    fit = [cv2.resize(i, (max(1, int(i.shape[1] * h / i.shape[0])), h)) for i in imgs]
    w = max(f.shape[1] for f in fit)
    pad = [np.pad(f, ((0, 0), (0, w - f.shape[1]), (0, 0)), constant_values=255) for f in fit]
    return Image.fromarray(cv2.cvtColor(np.hstack(pad), cv2.COLOR_BGR2RGB))

bad = 0
for vp in sorted(glob.glob('validation/*__BC.jpg')):
    stem = os.path.basename(vp)[:-len('__BC.jpg')]
    src = f'run/refs/{stem}__bald.jpg'
    if not os.path.exists(src):
        continue
    a = cv2.imread(vp); b, _ = C.crop_bc(cv2.imread(src), f'val_{stem}')
    if abs(a.shape[0] - b.shape[0]) > 8 or abs(a.shape[1] - b.shape[1]) > 8:
        print(f'  validate {stem}: SHAPE {a.shape[:2]} vs {b.shape[:2]}'); bad += 1; continue
    mad = float(np.abs(a.astype(np.float32) - cv2.resize(b, (a.shape[1], a.shape[0])).astype(np.float32)).mean())
    print(f'  validate {stem}: MAD {mad:.2f}')
    bad += mad > 4.0
assert not bad, 'the cropper does not reproduce the refs of record on this machine'

import garment_crop as GC, time
print('BiRefNet provider:', GC._STATE.get('biref_prov', '(not loaded)'),
      '| V2_ORT_GPU =', os.environ.get('V2_ORT_GPU'))

g = sorted(garments)[0]
small = cv2.imread(f'run/refs/{g}__VEi_small.jpg')
t0 = time.time(); cut, cranium = C.crop_bc(small, f'probe_{g}'); secs = time.time() - t0
print(f'{g}: {small.shape[1]}x{small.shape[0]} -> {cut.shape[1]}x{cut.shape[0]}  '
      f'cranium_used={cranium}  {secs:.1f}s  (CPU-only takes ~80s an image)')
display(side_by_side([cv2.imread(f'run/inputs/{g}__A4.jpg'), small, cut]))
assert cranium, 'the parser did not fire on the mannequin head - stop and look before running the arm'


In [ ]:
# 7 · one pair end to end, before paying for the set
import run_v35_linkC as V
V.main(MATRIX, 'testset', seeds=SEEDS[:1], arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR, limit=1)
print(sorted(f for f in os.listdir('run/gen') if '__VEic__' in f or '__M1qc__' in f))

In [ ]:
# 8 · the run: 23 references + 186 edits (resumable - rerun after any disconnect)
import run_v35_linkC as V, json
V.main(MATRIX, 'testset', seeds=SEEDS, arms=ARMS, gpu_usd_per_hour=A100_USD_PER_HOUR)
print(json.dumps(json.load(open('run/meta/cost_v35.json')), indent=1))

In [ ]:
# 9 · check every cell landed, the parser fired, and where the ankle cut was a no-op
import json, glob
meta = json.load(open('run/meta/prompts_v35.json'))
missed = [g for g in sorted(garments) for b in ('VEi', 'M1q')
          if any(a.startswith(b) for a in ARMS) and not meta.get(g, {}).get(f'{b}_cranium_used')]
print('references where the parser did NOT fire:', missed or 'none')
noop = [g for g in sorted(garments) for a in ARMS
        if a.endswith('a') and meta.get(g, {}).get(f'{a}_ankle_row') is None]
print(f'ankle cut a no-op (no ankles in frame) on {len(noop)} of '
      f'{len(garments) * sum(a.endswith("a") for a in ARMS)} cut references')
for a in list(ARMS) + ['VEi', 'BC']:
    n = len(glob.glob(f'run/gen/*__{a}__*.jpg'))
    print(f'  {a:6s} {n}/{len(rows) * len(SEEDS)} cells')
assert not missed, 'a reference fell back off the parser - its head crop is not comparable'

In [ ]:
# 10 · zip references, outputs and meta to Drive
import shutil, time, zipfile
name = f"v35_linkC_{time.strftime('%Y%m%d_%H%M')}"
KEEP = ('__VEic', '__M1qc', '__VEica', '__M1qca', '__VEi_headcut', '__M1q_headcut',
        '__M1q_small', '__VEi_small', '__BC')
with zipfile.ZipFile(f'/content/{name}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir('run/refs'):
        if any(t in f for t in KEEP): z.write('run/refs/' + f, 'refs/' + f)
    for f in os.listdir('run/gen'):    z.write('run/gen/' + f, 'gen/' + f)
    for f in os.listdir('run/inputs'): z.write('run/inputs/' + f, 'inputs/' + f)
    for f in os.listdir('run/meta'):   z.write('run/meta/' + f, 'meta/' + f)
os.makedirs(os.path.join(BASE, 'v3_runs'), exist_ok=True)
shutil.copy(f'/content/{name}.zip', os.path.join(BASE, 'v3_runs', f'{name}.zip'))
print(name, round(os.path.getsize(f'/content/{name}.zip') / 1e6, 1), 'MB -> Drive v3_runs/')